# Brain Extraction in Neurodesk: FSL BET, ANTs, and HD-BET

**Author**: Michèle Masson-Trottier

The University of Queensland<br>
<div style="line-height: 2;">
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0
    </a>
</div>

## Purpose

Brain extraction (skull stripping) removes non-brain tissue from T1-weighted MRI images. This is a critical preprocessing step for most neuroimaging pipelines. Neurodesk provides several tools with different strengths: **FSL BET** (fast, widely used), **ANTs BrainExtraction** (accurate, atlas-based), and **HD-BET** (deep learning, robust to non-standard MRI).

This tutorial compares all three approaches and demonstrates how to choose the appropriate method for your data.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Load and run FSL BET for fast skull stripping
- Run ANTs BrainExtraction with a brain template
- Run HD-BET for deep-learning-based extraction
- Compare outputs visually in FSLeyes
- Choose the appropriate method for your data
:::

## Citation and Resources

### Tools used in this workflow

__FSL BET__
: Smith, S.M. (2002). Fast robust automated brain extraction. *Human Brain Mapping*, 17(3), 143–155. [https://doi.org/10.1002/hbm.10062](https://doi.org/10.1002/hbm.10062)

__ANTs BrainExtraction__
: Avants, B.B., Tustison, N.J., Song, G., et al. (2011). A reproducible evaluation of ANTs similarity metric performance in brain image registration. *NeuroImage*, 54(3), 2033–2044. [https://doi.org/10.1016/j.neuroimage.2010.09.025](https://doi.org/10.1016/j.neuroimage.2010.09.025)

__HD-BET__
: Isensee, F., Schell, M., Pflueger, I., et al. (2019). Automated Brain Extraction of Multisequence MRI Using Artificial Neural Networks. *Human Brain Mapping*, 40(17), 4952–4964. [https://doi.org/10.1002/hbm.24750](https://doi.org/10.1002/hbm.24750)

### Educational resources

- [FSL Brain Extraction Tool documentation](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/BET)
- [ANTs documentation](http://stnava.github.io/ANTs/)
- [HD-BET on GitHub](https://github.com/MIC-DKFZ/HD-BET)
- [Neurodesk documentation](https://neurodesk.org)

## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment
- [ ] Familiarity with basic terminal commands
- [ ] No additional installation required — FSL, ANTs, and HD-BET are all pre-installed in Neurodesk
- [ ] The ability to visualise NIfTI images (FSLeyes will be used in this tutorial)

## Overview: Choosing a Brain Extraction Method

Each brain extraction tool has different strengths. Use this table to decide which method is best for your data:

| Method | Speed | Accuracy | GPU needed | Best for |
|--------|-------|----------|------------|----------|
| **FSL BET** | Very fast | Good | No | Quick quality control, standard T1w images |
| **ANTs BrainExtraction** | Slow | Excellent | No | Research pipelines requiring high accuracy |
| **HD-BET** | Fast | Excellent | Optional | Challenging data, non-standard MRI sequences |

All three methods are available in Neurodesk. We recommend starting with **FSL BET** for speed, and using **HD-BET** if the result is unsatisfactory.

## Download Sample Data

Open a terminal in Neurodesk and run the following commands to download a sample T1-weighted image from the OpenNeuro dataset ds000102.

:::{tip}
To paste in Neurodesk, use right-click with your mouse!
:::

```bash
cd ~/neurodesktop-storage/
datalad install https://github.com/OpenNeuroDatasets/ds000102.git
cd ds000102
datalad get sub-08/anat/sub-08_T1w.nii.gz
mkdir -p ~/neurodesktop-storage/brain_extraction/
cp sub-08/anat/sub-08_T1w.nii.gz ~/neurodesktop-storage/brain_extraction/T1w.nii.gz
```

![Terminal output from datalad download](/static/tutorials/structural_imaging/brain_extraction/terminal_datalad_download.png)
*Terminal output showing successful download of the sample T1w image.*

## Method 1: FSL BET

**FSL BET** (Brain Extraction Tool) is the fastest method and is ideal for quick quality control. Load FSL and run BET with the `-R` (robust) flag:

```bash
ml fsl/6.0.7.8
mkdir -p ~/neurodesktop-storage/brain_extraction/bet/
bet ~/neurodesktop-storage/brain_extraction/T1w.nii.gz \
    ~/neurodesktop-storage/brain_extraction/bet/T1w_brain.nii.gz \
    -R -f 0.5 -g 0 -m
```

### Understanding BET parameters

- `-R` : Robust brain extraction (better for problematic images)
- `-f 0.5` : Fractional intensity threshold (default 0.5; lower values = larger brain mask, higher = smaller)
- `-g 0` : Vertical gradient in fractional intensity threshold (for non-uniform intensity)
- `-m` : Generate a binary brain mask as well as the extracted brain

If the result looks too small or too large, adjust the `-f` parameter:
- Try `-f 0.3` or `-f 0.4` for a larger mask
- Try `-f 0.6` or `-f 0.7` for a smaller mask

![FSL BET brain extraction result in FSLeyes](/static/tutorials/structural_imaging/brain_extraction/bet_output_fsleyes.png)
*FSLeyes showing BET brain extraction result with brain mask overlay.*

## Method 2: ANTs BrainExtraction

**ANTs BrainExtraction** uses atlas-based registration for highly accurate skull stripping. It requires a brain template and associated probability mask. Load ANTs and download the OASIS-30 template:

```bash
ml ants/2.5.3
mkdir -p ~/neurodesktop-storage/brain_extraction/ants/
mkdir -p ~/neurodesktop-storage/templates/

# Download the OASIS-30 template
cd ~/neurodesktop-storage/templates/
wget https://dl.dropboxusercontent.com/s/aagvt0l0fyhwhw9/OASIS-30_Atropos_template.tar.gz
tar -xzf OASIS-30_Atropos_template.tar.gz
```

Then run antsBrainExtraction:

```bash
antsBrainExtraction.sh \
    -d 3 \
    -a ~/neurodesktop-storage/brain_extraction/T1w.nii.gz \
    -e ~/neurodesktop-storage/templates/OASIS-30_Atropos_template/T_template0.nii.gz \
    -m ~/neurodesktop-storage/templates/OASIS-30_Atropos_template/T_template0_BrainCerebellumProbabilityMask.nii.gz \
    -o ~/neurodesktop-storage/brain_extraction/ants/T1w_ants_
```

### Understanding antsBrainExtraction parameters

- `-d 3` : 3D image
- `-a` : Input anatomical image
- `-e` : Brain template (OASIS-30 is standard)
- `-m` : Brain mask template
- `-o` : Output prefix

This method is slower but produces very accurate results, especially for challenging images.

![ANTs BrainExtraction result in FSLeyes](/static/tutorials/structural_imaging/brain_extraction/ants_output_fsleyes.png)
*FSLeyes showing ANTs BrainExtraction result with high anatomical accuracy.*

## Method 3: HD-BET

**HD-BET** uses deep learning (a trained neural network) for robust, fast brain extraction. It is particularly useful for non-standard MRI sequences or challenging anatomy. Load the HD-BET container and run:

```bash
ml hdbet/1.0.0
mkdir -p ~/neurodesktop-storage/brain_extraction/hdbet/
hd-bet -i ~/neurodesktop-storage/brain_extraction/T1w.nii.gz \
       -o ~/neurodesktop-storage/brain_extraction/hdbet/T1w_brain.nii.gz
```

### Understanding HD-BET

- `-i` : Input image
- `-o` : Output brain image
- By default, HD-BET uses the CPU (faster in containers without GPU)
- No additional parameters are needed; HD-BET handles everything automatically

HD-BET is excellent for problematic cases because the neural network has learned from thousands of diverse brain images.

![HD-BET brain extraction result in FSLeyes](/static/tutorials/structural_imaging/brain_extraction/hdbet_output_fsleyes.png)
*FSLeyes showing HD-BET result, which is robust to MRI variations.*

## Comparing Results in FSLeyes

Now let's load all three brain masks and compare them visually. Open FSLeyes and load the original T1w image with all three brain masks as overlays:

```bash
ml fsl/6.0.7.8
fsleyes ~/neurodesktop-storage/brain_extraction/T1w.nii.gz \
        ~/neurodesktop-storage/brain_extraction/bet/T1w_brain_mask.nii.gz \
        ~/neurodesktop-storage/brain_extraction/ants/T1w_ants_BrainExtractionMask.nii.gz \
        ~/neurodesktop-storage/brain_extraction/hdbet/T1w_brain_mask.nii.gz &
```

In FSLeyes:
1. Set different colours for each mask (e.g., red for BET, blue for ANTs, green for HD-BET)
2. Adjust the overlay opacity to see where masks differ
3. Look at edges of the brain (cerebellum, brainstem) to spot differences
4. Check if any masks include non-brain tissue (e.g., dura, CSF spaces)

![FSLeyes comparison of brain masks](/static/tutorials/structural_imaging/brain_extraction/comparison_fsleyes.png)
*FSLeyes comparison of BET (red), ANTs (blue), and HD-BET (green) brain masks overlaid on the T1w image.*

## Summary

In this tutorial you:

1. Downloaded a sample T1-weighted MRI image from OpenNeuro
2. Extracted the brain using three different methods: FSL BET, ANTs BrainExtraction, and HD-BET
3. Compared the outputs visually in FSLeyes

### When to use each method:

- **FSL BET**: When you need a quick result for quality control or have standard T1w images with good contrast
- **ANTs BrainExtraction**: When you need the highest accuracy for research pipelines, even if it takes longer
- **HD-BET**: When you have challenging images, non-standard sequences, or are processing a large dataset that requires robust, automated extraction

All extracted brains and masks can now be used as inputs for downstream preprocessing steps such as tissue segmentation, registration, or surface extraction.

:::{seealso}
- [Visualising Neuroimaging Data with FSLeyes](fsleyes.ipynb) — for detailed guidance on inspecting brain extraction results
- [Structural Connectivity Analysis](structuralconnectivity.ipynb) — a downstream analysis using brain-extracted images
- [T1w Image Processing](t1_processing.ipynb) — full preprocessing pipeline including brain extraction
:::